## Data download: SARB Treasury bill tender rates

Daily 91, 182, 273 and 364-day Treasury bill tender rates are downloaded from the SARB web data service, one calendar year at a time, and saved unchanged to `Data/Raw/`.

In [1]:
# Import the requests library, which lets Python download data from a web address
import requests
# Import pandas, the main Python library for working with tables of data
import pandas as pd

# Web address of the SARB data service; the series code and dates get added to the end of it
BASE_URL = "https://custom.resbank.co.za/SarbWebApi/WebIndicators/Shared/GetTimeseriesObservations"

# Dictionary that links each SARB series code to a short, readable name
TBILL_CODES = {
    "MMRD203A": "tbill_91d",   # 91-day Treasury bill tender rate
    "MMRD206A": "tbill_182d",  # 182-day Treasury bill tender rate
    "MMRD209A": "tbill_273d",  # 273-day Treasury bill tender rate
    "MMRD212A": "tbill_364d",  # 364-day Treasury bill tender rate
}

# First calendar year to request (matches the start of the bond yield file)
START_YEAR = 1990
# Last calendar year to request
END_YEAR = 2026

# Empty list that will collect one small table for each series and each year
chunks = []

# Loop over every series code together with its readable name
for code, name in TBILL_CODES.items():
    # Loop over each year separately, because the service returns at most about 500 rows per request
    for year in range(START_YEAR, END_YEAR + 1):
        # Build the full web address for this series, from 1 January to 31 December of this year
        url = f"{BASE_URL}/{code}/{year}-01-01/{year}-12-31"
        # Download the data, and give up if the server has not answered within 60 seconds
        response = requests.get(url, timeout=60)
        # Stop with an error message if the download failed
        response.raise_for_status()
        # Convert the downloaded text (which is in JSON format) into a Python list of records
        records = response.json()
        # If this series has no data for this year, skip ahead to the next year
        if len(records) == 0:
            continue
        # Turn the list of records into a table
        df_year = pd.DataFrame(records)
        # Add a column that holds the SARB series code
        df_year["Code"] = code
        # Add a column that holds our short readable name
        df_year["Series"] = name
        # Add this year's table to the collection
        chunks.append(df_year)

# Stack all the small tables on top of each other to form one long table
tbills_raw = pd.concat(chunks, ignore_index=True)
# Keep only the useful columns, in a sensible order
tbills_raw = tbills_raw[["Code", "Series", "Timeseries", "Description", "Period", "Value"]]
# Sort the rows by series code and then by date
tbills_raw = tbills_raw.sort_values(["Code", "Period"]).reset_index(drop=True)

# Save the downloaded data, unchanged, as a CSV file in the Data/Raw folder
tbills_raw.to_csv("../Data/Raw/SARB_treasury_bills_daily.csv", index=False)

# Print the number of observations and the first and last date for each series
print(tbills_raw.groupby("Code")["Period"].agg(["count", "min", "max"]))

          count                  min                  max
Code                                                     
MMRD203A  10641  1990-01-02T00:00:00  2026-09-23T00:00:00
MMRD206A   6997  2002-01-02T00:00:00  2026-09-23T00:00:00
MMRD209A   5472  2007-02-13T00:00:00  2026-09-23T00:00:00
MMRD212A   4454  2010-06-30T00:00:00  2026-09-23T00:00:00
